In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, StackingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error,r2_score, mean_absolute_error

In [ ]:
class RegressaoComNormalizacao:
   def __init__(self, coluna_alvo, colunas_preditoras):
       self.coluna_alvo = coluna_alvo
       self.colunas_preditoras = colunas_preditoras
       self.scaler = StandardScaler()
       
       gbr = GradientBoostingRegressor(
           n_estimators=5000,
           learning_rate=0.02,
           max_depth=4,
           subsample=0.8,
           min_samples_split=15,
           min_samples_leaf=5,
           random_state=42,
           loss='huber'
       )
       
       xgbr = XGBRegressor(
           n_estimators=5000,
           learning_rate=0.02,
           max_depth=4,
           subsample=0.8,
           colsample_bytree=0.8,
           min_child_weight=2,
           objective='reg:squarederror',
           random_state=42,
           n_jobs=-1
       )
       
       rf = RandomForestRegressor(
           n_estimators=5000,
           max_depth=8,
           min_samples_split=15,
           min_samples_leaf=5,
           max_features='sqrt',
           bootstrap=True,
           random_state=42,
           n_jobs=-1
       )
       
       self.stack_regressor = StackingRegressor(
           estimators=[('gbr', gbr), ('xgbr', xgbr), ('rf', rf)],
           final_estimator=RidgeCV(),
           cv=5
       )

   def fit(self, X, y):
       self.y_mean = np.mean(y)
       self.y_std = np.std(y)
       
       X_scaled = self.scaler.fit_transform(X)
       self.stack_regressor.fit(X_scaled, y)
       return self

   def predict(self, X):
       X_scaled = self.scaler.transform(X)
       
       previsao_normalizada = self.stack_regressor.predict(X_scaled)
       
       previsao_original = previsao_normalizada
       
       return previsao_original

def regressao_avancada(df, coluna_alvo, colunas_preditoras, 
                       taxa_mascaramento=0.1, 
                       prever_novos_dados=None):
   np.random.seed(42)
   
   df = df.copy()
   
   valores_originais = df[coluna_alvo].copy()
   
   mask = np.random.rand(len(df)) < taxa_mascaramento
   
   X = df[colunas_preditoras]
   y = df[coluna_alvo]
   
   modelo = RegressaoComNormalizacao(coluna_alvo, colunas_preditoras)
   modelo.fit(X, y)
   
   if prever_novos_dados is not None:
       return modelo.predict(prever_novos_dados[colunas_preditoras])
   
   valores_mascarados = y.copy()
   valores_mascarados[mask] = modelo.predict(X[mask])
   
   rmse = np.sqrt(mean_squared_error(valores_originais[mask], valores_mascarados[mask]))
   r2 = r2_score(valores_originais[mask], valores_mascarados[mask])
   mae = mean_absolute_error(valores_originais[mask], valores_mascarados[mask])
   
   print(f"RMSE: {rmse:.4f}")
   print(f"R² Score: {r2:.4f}")
   print(f"MAE: {mae:.4f}")
   
   return (
       valores_mascarados, 
       rmse,  
       valores_mascarados[mask].tolist(),  
       valores_originais[mask].tolist() 
   )

Rgressão - sem timestamp


In [ ]:
#df = pd.read_csv('../datasets/arquivo-completo/completo vazao atraso traceroute 14-07-2024 3horas.csv')
#df = pd.read_csv('../datasets/arquivo-completo/completo vazao atraso traceroute 14-07-2024 30min.csv')
df = pd.read_csv('../datasets/arquivo-completo/completo vazao atraso traceroute 14-07-2024 10min.csv')
#df= pd.read_csv('../datasets/arquivo-completo/completo vazao atraso traceroute 14-07-2024 6horas.csv')
df

In [ ]:
colunas_preditoras = ['Vazao_bbr'] #, 'Hop_count', 'Bottleneck','Timestamp_cubic', , 'Atraso(ms)'
teste = regressao_avancada(df, 'Vazao', colunas_preditoras)

In [ ]:
teste[2]